# Robotics Harness Optimization with RHO

RHO improves robot policy **source code**, not model weights. The loop is intentionally simple:

1. Evaluate a seed program.
2. Let a coding agent propose a source edit.
3. Keep the edit only if its training reward improves.
4. Check the accepted program on validation.

This notebook has two short parts:

- **Part A:** run one live cube-stack repair with Gemma E2B.
- **Part B:** inspect a recorded two-task Qwen evolution.

Part B stops at the train and validation evidence already produced by HELIX; it adds no extra evaluation suite.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(r"""
## Part A — one live repair

The seed is an authentic failed Gemma E4B cube-stack program. Gemma E2B gets one bounded attempt to edit `solver/`; the evaluator and HELIX configuration stay protected.

We will inspect only the essentials: the seed, fixed budgets, training-gate improvement, validation check, accepted diff, and video paths.
"""))

In [ ]:
import json
import sys
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import Markdown, Video, display

sys.path.insert(0, "/ryzers/notebooks/scripts")
import rho_demo

FAST_ROOT = Path("/tmp/rho_fast_notebook")
FAST_TIMEOUT_SECONDS = 480
rho_demo.VIDEO_ROOT = FAST_ROOT / "videos"

if rho_demo.MODEL != rho_demo.DEFAULT_RHO_MODEL:
    raise RuntimeError("Part A requires Gemma E2B; unset RHO_MODEL and restart.")

print("Mutation model:", rho_demo.DEFAULT_RHO_MODEL)
print("Budget: 1 generation · 1 proposal ·", FAST_TIMEOUT_SECONDS, "second timeout")

In [ ]:
display(Markdown(r"""
### What HELIX does here

`seed diagnostics → one source edit → strict training gate → validation check`

Only `solver/` is editable. A changed file is not automatically a success: the training reward must improve before the candidate is retained.
"""))

In [ ]:
setup_started = perf_counter()
rho_demo.ensure_services(model=rho_demo.DEFAULT_RHO_MODEL)
FAST_REPO = rho_demo.prepare_workshop(FAST_ROOT / "candidate", generations=1)
FAST_SETUP_SECONDS = perf_counter() - setup_started

print(f"Setup: {FAST_SETUP_SECONDS:.1f}s")
print("Seed repository:", FAST_REPO)
print("Editable: solver/program.py and solver/policy.py")
print("Protected evaluator:", FAST_REPO / "probe.py")

In [ ]:
print("===== failed seed =====")
print((FAST_REPO / "solver" / "program.py").read_text())

print("===== fixed HELIX settings =====")
for line in (FAST_REPO / "helix.toml").read_text().splitlines():
    if any(key in line for key in (
        "max_generations", "max_evaluations", "acceptance_criterion",
        "num_parallel_proposals", "timeout_seconds",
    )):
        print(line)

In [ ]:
display(Markdown(r"""
### Before the mutation

The training trial controls acceptance. The separate validation trial is a quick transfer check. Exceptions and timeouts force deployable reward to zero.
"""))

In [ ]:
def result_row(label, result):
    return {
        "phase": label,
        "trial": result.get("trial"),
        "reward": result.get("reward"),
        "raw reward": result.get("raw_reward"),
        "completed": result.get("task_completed"),
        "timed out": result.get("timed_out"),
    }


fast_baseline_started = perf_counter()
FAST_BEFORE_TRAIN = rho_demo.score_candidate(FAST_REPO, "train")
FAST_BEFORE_VAL = rho_demo.score_candidate(FAST_REPO, "val", capture=True)
FAST_BASELINE_SECONDS = perf_counter() - fast_baseline_started

display(pd.DataFrame([
    result_row("train", FAST_BEFORE_TRAIN),
    result_row("validation", FAST_BEFORE_VAL),
]).set_index("phase"))
print(FAST_BEFORE_TRAIN["feedback"][-800:])

In [ ]:
fast_evolution_started = perf_counter()
FAST_RUN = rho_demo.run_helix(
    FAST_REPO,
    generations=1,
    timeout_seconds=FAST_TIMEOUT_SECONDS,
)
FAST_EVOLUTION_SECONDS = perf_counter() - fast_evolution_started
FAST_SUMMARY = rho_demo.summarize_run(FAST_REPO)
FAST_BEST = Path(FAST_SUMMARY["live_best"])
FAST_AFTER_TRAIN = rho_demo.score_candidate(FAST_BEST, "train")
FAST_AFTER_VAL = rho_demo.score_candidate(FAST_BEST, "val", capture=True)

print("Accepted:", FAST_SUMMARY["accepted"], "· timed out:", FAST_RUN.timed_out)
print("\n===== training gate: mutation improvement =====")
display(pd.DataFrame([
    result_row("seed", FAST_BEFORE_TRAIN),
    result_row("mutated", FAST_AFTER_TRAIN),
]).set_index("phase"))

print("===== separate validation check =====")
display(pd.DataFrame([
    result_row("seed", FAST_BEFORE_VAL),
    result_row("mutated", FAST_AFTER_VAL),
]).set_index("phase"))

print("\n===== accepted diff =====")
print(rho_demo.source_diff(FAST_REPO, FAST_BEST) or "No accepted source change.")

print("\n===== validation rollouts =====")
for label, result in (("Before mutation", FAST_BEFORE_VAL), ("After mutation", FAST_AFTER_VAL)):
    video_path = Path(result["video"]) if result.get("video") else None
    display(Markdown(f"**{label}**"))
    if video_path is not None and video_path.is_file():
        display(Video(str(video_path), embed=True, width=420, html_attributes="controls loop"))
    else:
        print("Video unavailable:", video_path)

### Part A takeaway

The first table shows the improvement HELIX actually gated on. The second asks whether that repair transfers to validation. Keep both results visible: training improvement does not guarantee validation success.

The disposable repository remains under `/tmp/rho_fast_notebook/` for inspection.

In [ ]:
display(Markdown(r"""
## Part B — recorded two-task evolution

Qwen3-Coder evolved two policy files over two generations:

- `solver/tasks/cube_stack.py`
- `solver/tasks/spill_wipe.py`

We will show only the candidate validation scores and the selected source diff.
"""))

### What changes in the multi-task case

A candidate can improve stack, wipe, both, or neither. HELIX therefore keeps task-level validation scores instead of reducing everything to one opaque number. The recorded table below is enough to see which candidates helped.

In [ ]:
RECORDED_REPORT_PATH = Path(
    "/ryzers/notebooks/recorded_results/rho_multitask_report.json"
)
if not RECORDED_REPORT_PATH.is_file():
    raise FileNotFoundError(f"Missing recorded report: {RECORDED_REPORT_PATH}")

LONG_REPORT = json.loads(RECORDED_REPORT_PATH.read_text())
assert LONG_REPORT["schema_version"] == "rho-multitask-helix-report/v2"

print("Mutation model:", LONG_REPORT["mutation_model_loader_alias"])
print("Generations:", LONG_REPORT["generations"])
print("Selected candidate:", LONG_REPORT["selected_candidate"])
print("Report:", RECORDED_REPORT_PATH)

### Read the validation table

`stack_val` and `wipe_val` are the two ordinary validation scenarios configured in HELIX. They explain candidate retention, but they are not an independent generalization test.

In [ ]:
LONG_FRONTIER = LONG_REPORT["frontier"]
LONG_SELECTED = LONG_REPORT["selected_candidate"]

display(pd.DataFrame([
    {
        "candidate": candidate_id,
        "stack validation": candidate["scores"].get("stack_val", 0.0),
        "wipe validation": candidate["scores"].get("wipe_val", 0.0),
        "retained": candidate["frontier"],
    }
    for candidate_id, candidate in LONG_FRONTIER["candidates"].items()
]).set_index("candidate"))

print("===== selected source diff =====")
print(LONG_REPORT["selected_diff"] or "No selected source change.")

In [ ]:
baseline_by_scenario = {
    result["scenario_id"]: result
    for result in LONG_REPORT["baseline_validation"]
}
selected_scores = LONG_FRONTIER["candidates"][LONG_SELECTED]["scores"]

comparison = []
for scenario_id, task in (
    ("stack_val", "cube_stack"),
    ("wipe_val", "spill_wipe"),
):
    before = baseline_by_scenario[scenario_id]
    comparison.append({
        "task": task,
        "seed validation reward": before["reward"],
        "selected validation reward": selected_scores[scenario_id],
    })

display(pd.DataFrame(comparison).set_index("task"))
print("These are the ordinary validation scores already used by HELIX.")

In [ ]:
print("Part A live timing")
print(f"  setup:      {FAST_SETUP_SECONDS:7.1f}s")
print(f"  baseline:   {FAST_BASELINE_SECONDS:7.1f}s")
print(f"  evolution:  {FAST_EVOLUTION_SECONDS:7.1f}s")

print("\nPart B recorded timing")
for key in ("setup_seconds", "baseline_seconds", "evolution_seconds"):
    print(f"  {key.replace('_', ' '):20s} {LONG_REPORT['timing'][key]:7.1f}s")

In [ ]:
rho_demo.stop_owned_services()
print("Notebook-owned model and robotics services stopped.")

## The whole idea

RHO wraps a coding agent in an evaluator:

**diagnostics → source mutation → strict training gate → validation evidence**

Part A shows the loop live on one file. Part B shows the same idea when two policy files have separate validation scores. The useful evidence is the diff plus measured reward—not merely that an agent changed code.

## What not to claim

The recorded Part B scores come from the same validation scenarios used by HELIX. They explain candidate selection, but they do not establish broad generalization.

Report the exact source diff, deployable reward, completion flag, and rejected mutations. Do not substitute mock output when a live mutation fails.